# Merchant Attrition Early-Warning

Build a **grounded, self-verifying** agent that flags **which merchants are decaying toward churn** by category — and contextualizes that decline against the category's **demand backdrop** (US Census retail sales), so you separate a real attrition problem from a whole-category slowdown.

| Step | What you build |
|---|---|
| 1 | Explore the benchmark (`ATTRITION_BENCHMARK`) |
| 2 | Query the **semantic view** (Cortex Analyst) |
| 3 | Build a verified **fact pack** |
| 4 | **Grounded draft** under DATA RULES |
| 5 | **Verify** claims vs evidence |
| 6 | Persist the save-list note |

**Prereq:** run `sql/setup.sql` (synthetic merchant book and demand backdrop; optionally `sql/census_backdrop.sql` for real US Census retail sales). Run the notebook as `MERCHANT_ATTRITION_ROLE` on `SNOWPACK_MERCHANT_ATTRITION_WH`. The steps match the app's Attrition notes screen (`app/ai.py`).

In [ ]:
SCHEMA = "SNOWPACK_MERCHANT_ATTRITION.CORE"
MODEL  = "claude-sonnet-4-5"
import json
from snowflake.snowpark.context import get_active_session
session = get_active_session()
mode = session.sql(f"select mode, note from {SCHEMA}.ATTRITION_DATA_SOURCE_INFO").collect()[0]
print("data mode:", mode[0], "—", mode[1])

## Step 1 — Explore the benchmark

`ATTRITION_BENCHMARK` = merchant health rolled up to **(category, month, risk_band, tenure_band)**, joined to the category demand backdrop. `risk_band` ∈ healthy / watch / at-risk / churned; NRR = volume retained vs baseline.

In [ ]:
session.sql(f"""
  SELECT category_name, risk_band,
         SUM(merchant_count)  AS merchants,
         ROUND(SUM(monthly_volume_usd)/1e6,1) AS volume_m,
         ROUND(AVG(avg_volume_decline_pct),1) AS avg_decline_pct,
         ROUND(AVG(category_yoy_growth_pct),1) AS cat_demand_yoy
  FROM {SCHEMA}.ATTRITION_BENCHMARK
  WHERE month_end = (SELECT MAX(month_end) FROM {SCHEMA}.ATTRITION_BENCHMARK)
  GROUP BY category_name, risk_band
  ORDER BY category_name, risk_band
""").to_pandas()

## Step 2 — Cortex Analyst over the semantic view

`ATTRITION_SV` lets Analyst answer NL questions like *"which categories have the most at-risk merchants in 2026?"*. Direct `SEMANTIC_VIEW()` query below; the agent writes this for you from plain English. `churn_rate_pct` and `net_revenue_retention` are percentages; category metrics are non-additive (AVG).

In [ ]:
session.sql(f"""
  SELECT * FROM SEMANTIC_VIEW({SCHEMA}.ATTRITION_SV
      METRICS at_risk_merchants, churn_rate_pct, net_revenue_retention, category_demand_yoy
      DIMENSIONS category_name, report_year)
  WHERE report_year = 2026 ORDER BY at_risk_merchants DESC
""").to_pandas()

## Step 3 — Verified fact pack

Latest-month health per category — the agent's only allowed numbers: active vs churned merchants, churn rate, NRR, volume decline, and the category demand backdrop.

In [ ]:
# Same fact pack as app/ai.py (scope: all categories): latest month and the same month a year earlier.
fact_pack = json.loads(session.sql(f"""
        WITH z AS (SELECT MAX(month_end) z FROM {SCHEMA}.ATTRITION_BENCHMARK),
        g AS (
          SELECT b.category_code, b.category_name, b.month_end,
                 SUM(b.merchant_count) AS total_merchants, SUM(b.active_merchant_count) AS active_merchants,
                 SUM(b.churned_count) AS churned_merchants, SUM(b.at_risk_count) AS at_risk_merchants,
                 SUM(b.watch_count) AS watch_merchants, SUM(b.monthly_volume_usd) AS volume,
                 SUM(b.baseline_volume_usd) AS baseline, AVG(b.category_yoy_growth_pct) AS cat_yoy
          FROM {SCHEMA}.ATTRITION_BENCHMARK b, z
          WHERE b.month_end IN (z.z, LAST_DAY(DATEADD(month, -12, z.z)))
          GROUP BY 1, 2, 3)
        SELECT ARRAY_AGG(OBJECT_CONSTRUCT(
          'category', l.category_name, 'month', l.month_end::string,
          'total_merchants', l.total_merchants, 'active_merchants', l.active_merchants,
          'watch_merchants', l.watch_merchants, 'at_risk_merchants', l.at_risk_merchants,
          'churned_merchants', l.churned_merchants,
          'churn_rate_pct', ROUND(l.churned_merchants * 100.0 / NULLIF(l.total_merchants, 0), 1),
          'net_revenue_retention_pct', ROUND(l.volume * 100.0 / NULLIF(l.baseline, 0), 1),
          'book_volume_usd', l.volume, 'book_volume_year_earlier_usd', y.volume,
          'book_volume_yoy_pct', ROUND((l.volume / NULLIF(y.volume, 0) - 1) * 100, 1),
          'churned_merchants_year_earlier', y.churned_merchants,
          'category_demand_yoy_pct', ROUND(l.cat_yoy, 1))) WITHIN GROUP (ORDER BY l.category_name)
        FROM g l JOIN z ON l.month_end = z.z
        JOIN g y ON y.category_code = l.category_code AND y.month_end <> z.z
        
    """).collect()[0][0])
print(json.dumps(fact_pack, indent=1)[:900], "...")


## Step 4 — Grounded draft (DATA RULES)

The model may use ONLY the fact pack — no invented numbers, no hedging.

In [ ]:
DATA_RULES = "You are a payments merchant-retention analyst. Write a concise attrition early-warning note using ONLY the FACTS JSON.\nDATA RULES: use only numbers in FACTS, copy exactly; churn_rate_pct, net_revenue_retention_pct and the *_yoy_pct fields are percentages; book_volume_yoy_pct is the merchant book's processed-volume change vs the same month a year earlier; category_demand_yoy_pct is the category demand backdrop: contrast the book with it, and call a category a retention problem when its book shrinks much faster than its demand, or a market slowdown when demand itself is shrinking; dollars are USD; no estimates, no hedging, no outside benchmarks; missing = n/d. Do not name any real company, processor or merchant. Write in plain English only.\nOUTPUT markdown starting with '# ', a one-line takeaway (where attrition is worst), a table (category | active | watch | at-risk | churned | churn rate % | NRR % | book YoY % | category demand YoY %), 3 bullets prioritizing a save-list, ending with '## Data Coverage'."

def complete(prompt):
    raw = session.sql("SELECT AI_COMPLETE(?, ?)", params=[MODEL, prompt]).collect()[0][0]
    # The string result can arrive JSON-encoded ("..." with escapes); decode it once.
    if isinstance(raw, str) and raw.startswith('"'):
        try:
            raw = json.loads(raw)
        except ValueError:
            pass
    return raw.strip()

draft_md = complete(DATA_RULES + f"\n\nFACTS: {json.dumps(fact_pack)}")
print(draft_md)


## Step 5 — Verify against the evidence

A second pass fact-checks every number against the fact pack and flags hedges — the quality gate.

In [ ]:
fp = fact_pack
draft = draft_md
verify_prompt = (
        "You are a fact-checker. Compare the DRAFT against the FACTS JSON (only allowed source). Return ONLY a JSON object with keys "
        "status (exactly 'verified' if every number/claim is supported and no hedge words like likely/approximately/around/could, else exactly 'needs_review'), "
        "residual_hedges (int), edits (array of strings, each naming the unsupported or hedged claim). "
        "Exact figures written with thousands separators, and rounding a FACTS number to the draft's display units ($M/$K with one or two decimals, whole or one-decimal "
        "percentages) counts as supported, and so do differences of two FACTS numbers (e.g. percentage-point gaps) when the "
        "arithmetic is right, rounded to whole or one-decimal points. The draft was told to label categories 'retention problem' "
        "(book shrinking much faster than category demand) or 'market slowdown' (demand itself shrinking) and to recommend "
        "save-list priorities: such labels and recommendations are allowed when they follow from FACTS. Flag only wrong numbers, "
        "invented facts, hedges and any non-English text.\n\n"
        f"FACTS: {json.dumps(fp)}\n\nDRAFT:\n{draft}"
)
raw = complete(verify_prompt)
import re as _re
def _parse_verdict(x):
    t = x.strip()
    for _ in range(3):
        m = _re.match(r"^```[a-zA-Z]*\s*(.*?)\s*```$", t, _re.S)
        if m:
            t = m.group(1).strip()
        try:
            v = json.loads(t)
        except Exception:
            a, b = t.find("{"), t.rfind("}")
            if a >= 0 and b > a:
                try:
                    v = json.loads(t[a:b + 1])
                except Exception:
                    return None
            else:
                return None
        if isinstance(v, str):
            t = v.strip()
            continue
        return v
    return None
verdict = _parse_verdict(raw) or {"status": "needs_review", "residual_hedges": None, "edits": ["unparseable verifier output"]}
status = verdict.get("status", "needs_review")
edits = verdict.get("edits", []) or []
hedges = verdict.get("residual_hedges")
print(f"STATUS: {status.upper()}   residual_hedges={hedges}   edits={len(edits)}")
for e in edits:
    print(" -", e)

## Step 6 — Persist the note

In [ ]:
session.sql(
    f"INSERT INTO {SCHEMA}.ATTRITION_REPORTS (subject, draft_md, report_md, status, edits_applied, residual_hedges) SELECT ?, ?, ?, ?, ?, ?",
    params=["Merchant attrition early-warning", draft_md, draft_md, status, len(edits), hedges]
).collect()
session.sql(f"SELECT run_id, subject, status, edits_applied, residual_hedges, created_at FROM {SCHEMA}.ATTRITION_REPORTS ORDER BY created_at DESC LIMIT 5").to_pandas()

## Recap & extensions

You built the verified attrition early-warning loop on free category-demand data. `setup.sql` already created an **`ATTRITION_AGENT`** — chat with it in Snowsight → AI & ML → Agents (*"which at-risk merchants should we save first, and is their category still growing?"*).

**Next:** point `ATTRITION_MERCHANT` / `ATTRITION_MERCHANT_MONTHLY` at your own merchant master + settlement/engagement data (same columns) to run this on your real book.

**Try breaking the verifier:** add a made-up projection (e.g., *"restaurants will likely lose 20% more merchants next quarter"*) to the draft and watch Step 5 flag it *Needs review*.